# Google Colab R

- Google is now supporting a Colab notebook with the R kernel
- We can run R codes directly on Colab
- R Colab Notebook Link: [R Google Colab](https://colab.fan/r)
    - Colab with R kernel
    - With base-R installed
    - Run R codes immediately
    - No need to setup
    - Save a copy in your Drive


## Demonstration

This notebook uses the R kernel.


In [ ]:
sessionInfo()

R version 4.0.2 (2020-06-22)
Platform: x86_64-pc-linux-gnu (64-bit)
Running under: Ubuntu 18.04.5 LTS

Matrix products: default
BLAS:   /usr/lib/x86_64-linux-gnu/openblas/libblas.so.3
LAPACK: /usr/lib/x86_64-linux-gnu/libopenblasp-r0.2.20.so

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

attached base packages:
[1] stats     graphics  grDevices utils     datasets  methods   base     

loaded via a namespace (and not attached):
 [1] compiler_4.0.2  ellipsis_0.3.1  IRdisplay_0.7.0 pbdZMQ_0.3-3   
 [5] tools_4.0.2     htmltools_0.5.0 pillar_1.4.6    base64enc_0.1-3
 [9] crayon_1.3.4    uuid_0.1-4      IRkernel_1.1.1  jsonlite_1.7.1 
[13] digest_0.6.25   lifecycle_0.2.0 repr_1.1

In [1]:

library(readxl)
library(readr)
library(dplyr)
library(lubridate)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




In [7]:
library(readr)
library(dplyr)
install.packages("tseries")
library(tseries)
library(knitr)

Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘xts’, ‘TTR’, ‘quadprog’, ‘zoo’, ‘quantmod’


Registered S3 method overwritten by 'quantmod':
  method            from
  as.zoo.data.frame zoo 



In [8]:
library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(knitr)
library(tibble)

In [9]:
library(readr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(knitr)

In [10]:
library(readr)
library(dplyr)
library(ggplot2)
library(knitr)
library(broom)

In [5]:
read_fred_monthly <- function(file_name, value_name) {
  data <- read_excel(file_name, sheet = "Monthly") %>%
    rename(
      date = observation_date,
      value = 2
    ) %>%
    mutate(
      date = as.Date(date),
      value = as.numeric(value)
    ) %>%
    rename(!!value_name := value)

  return(data)
}

retail <- read_fred_monthly("/content/RSXFS.xlsx", "rsxfs")
cpi <- read_fred_monthly("/content/CPIAUCSL 3.xlsx", "cpi")
sentiment <- read_fred_monthly("/content/UMCSENT.xlsx", "umcsent")

final_data <- retail %>%
  inner_join(cpi, by = "date") %>%
  inner_join(sentiment, by = "date") %>%
  arrange(date) %>%
  mutate(
    days_in_month = lubridate::days_in_month(date),
    retail_per_day = rsxfs / days_in_month,
    cpi_base_jan2025 = cpi[date == as.Date("2025-01-01")][1],
    real_retail_per_day = retail_per_day * (cpi_base_jan2025 / cpi),
    log_real_retail_per_day = log(real_retail_per_day),
    retail_growth_pct = 100 * (log_real_retail_per_day - lag(log_real_retail_per_day)),
    inflation_pct = 100 * (log(cpi) - lag(log(cpi))),
    sentiment_change = umcsent - lag(umcsent)
  )

write_csv(final_data, "final_fred_monthly_data.csv")

glimpse(final_data)
summary(final_data)
head(final_data)
tail(final_data)

Rows: 411
Columns: 12
$ date                    <date> 1992-01-01, 1992-02-01, 1992-03-01, 1992-04-0…
$ rsxfs                   <dbl> 142419, 142584, 142120, 143659, 144239, 145273…
$ cpi                     <dbl> 138.3, 138.6, 139.1, 139.4, 139.7, 140.1, 140.…
$ umcsent                 <dbl> 67.5, 68.8, 76.0, 77.2, 79.2, 80.4, 76.6, 76.1…
$ days_in_month           <int> 31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31…
$ retail_per_day          <dbl> 4594.161, 4916.690, 4584.516, 4788.633, 4652.8…
$ cpi_base_jan2025        <dbl> 318.961, 318.961, 318.961, 318.961, 318.961, 3…
$ real_retail_per_day     <dbl> 10595.50, 11314.81, 10512.45, 10956.87, 10623.…
$ log_real_retail_per_day <dbl> 9.268185, 9.333868, 9.260316, 9.301722, 9.2708…
$ retail_growth_pct       <dbl> NA, 6.5682409, -7.3551915, 4.1406088, -3.09103…
$ inflation_pct           <dbl> NA, 0.21668481, 0.36010122, 0.21543994, 0.2149…
$ sentiment_change        <dbl> NA, 1.3, 7.2, 1.2, 2.0, 1.2, -3.8, -0.5, -0.5,…


      date                rsxfs             cpi           umcsent      
 Min.   :1992-01-01   Min.   :142120   Min.   :138.3   Min.   : 50.00  
 1st Qu.:2000-07-16   1st Qu.:243178   1st Qu.:172.7   1st Qu.: 73.90  
 Median :2009-02-01   Median :320312   Median :215.0   Median : 87.60  
 Mean   :2009-01-30   Mean   :341256   Mean   :215.3   Mean   : 84.52  
 3rd Qu.:2017-08-16   3rd Qu.:409370   3rd Qu.:244.9   3rd Qu.: 95.10  
 Max.   :2026-03-01   Max.   :651843   Max.   :330.3   Max.   :112.00  
                                       NAs    :1                       
 days_in_month   retail_per_day  cpi_base_jan2025 real_retail_per_day
 Min.   :28.00   Min.   : 4585   Min.   :319      Min.   :10512      
 1st Qu.:30.00   1st Qu.: 7902   1st Qu.:319      1st Qu.:14370      
 Median :31.00   Median :10547   Median :319      Median :15920      
 Mean   :30.44   Mean   :11221   Mean   :319      Mean   :16004      
 3rd Qu.:31.00   3rd Qu.:13607   3rd Qu.:319      3rd Qu.:17651      
 Max

date,rsxfs,cpi,umcsent,days_in_month,retail_per_day,cpi_base_jan2025,real_retail_per_day,log_real_retail_per_day,retail_growth_pct,inflation_pct,sentiment_change
<date>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1992-01-01,142419,138.3,67.5,31,4594.161,318.961,10595.50,9.268185,NA,NA,NA
1992-02-01,142584,138.6,68.8,29,4916.690,318.961,11314.81,9.333868,6.568241,0.2166848,1.3
1992-03-01,142120,139.1,76.0,31,4584.516,318.961,10512.45,9.260316,-7.355192,0.3601012,7.2
1992-04-01,143659,139.4,77.2,30,4788.633,318.961,10956.87,9.301722,4.140609,0.2154399,1.2
1992-05-01,144239,139.7,79.2,31,4652.871,318.961,10623.37,9.270811,-3.091038,0.2149768,2.0
1992-06-01,145273,140.1,80.4,30,4842.433,318.961,11024.61,9.307885,3.707372,0.2859187,1.2


date,rsxfs,cpi,umcsent,days_in_month,retail_per_day,cpi_base_jan2025,real_retail_per_day,log_real_retail_per_day,retail_growth_pct,inflation_pct,sentiment_change
<date>,<dbl>,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
2025-10-01,631346,NA,53.6,31,20366.00,318.961,NA,NA,NA,NA,-1.5
2025-11-01,634477,325.063,51.0,30,21149.23,318.961,20752.23,9.940409,NA,NA,-2.6
2025-12-01,634830,326.031,52.9,31,20478.39,318.961,20034.31,9.905202,-3.5207073,0.2973459,1.9
2026-01-01,634949,326.588,56.4,31,20482.23,318.961,20003.89,9.903682,-0.1519535,0.1706969,3.5
2026-02-01,639691,327.460,56.6,28,22846.11,318.961,22253.15,10.010239,10.6556788,0.2666473,0.2
2026-03-01,651843,330.293,53.3,31,21027.19,318.961,20305.77,9.918661,-9.1578435,0.8614229,-3.3


In [11]:
final_data <- read_csv("final_fred_monthly_data.csv")

glimpse(final_data)

Rows: 411 Columns: 12
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl  (11): rsxfs, cpi, umcsent, days_in_month, retail_per_day, cpi_base_jan2...
date  (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Rows: 411
Columns: 12
$ date                    <date> 1992-01-01, 1992-02-01, 1992-03-01, 1992-04-0…
$ rsxfs                   <dbl> 142419, 142584, 142120, 143659, 144239, 145273…
$ cpi                     <dbl> 138.3, 138.6, 139.1, 139.4, 139.7, 140.1, 140.…
$ umcsent                 <dbl> 67.5, 68.8, 76.0, 77.2, 79.2, 80.4, 76.6, 76.1…
$ days_in_month           <dbl> 31, 29, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31…
$ retail_per_day          <dbl> 4594.161, 4916.690, 4584.516, 4788.633, 4652.8…
$ cpi_base_jan2025        <dbl> 318.961, 318.961, 318.961, 318.961, 318.961, 3…
$ real_retail_per_day     <dbl> 10595.50, 11314.81, 10512.45, 10956.87, 10623.…
$ log_real_retail_per_day <dbl> 9.268185, 9.333868, 9.260316, 9.301722, 9.2708…
$ retail_growth_pct       <dbl> NA, 6.5682409, -7.3551915, 4.1406088, -3.09103…
$ inflation_pct           <dbl> NA, 0.21668481, 0.36010122, 0.21543994, 0.2149…
$ sentiment_change        <dbl> NA, 1.3, 7.2, 1.2, 2.0, 1.2, -3.8, -0.5, -0.5,…


In [18]:
final_data <- read_csv("final_fred_monthly_data.csv")

reg_data <- final_data %>%
  arrange(date) %>%
  mutate(
    inflation_lag1     = lag(inflation_pct, 1),
    sentiment_lag1     = lag(sentiment_change, 1),
    retail_growth_lag1 = lag(retail_growth_pct, 1)
  ) %>%
  select(
    date, retail_growth_pct,
    inflation_lag1, sentiment_lag1, retail_growth_lag1
  ) %>%
  na.omit()


Rows: 411 Columns: 12
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
dbl  (11): rsxfs, cpi, umcsent, days_in_month, retail_per_day, cpi_base_jan2...
date  (1): date

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [19]:
# Rolling Window CV

min_train_size <- 120   # minimum training window
test_size      <- 24    # forecast horizon
n_obs          <- nrow(reg_data)

# Storage for results from each rolling split
rolling_results <- data.frame()

# Number of possible origins
n_origins <- n_obs - min_train_size - test_size + 1

# Step through origins (use every 3rd to keep runtime reasonable)
origins <- seq(min_train_size, n_obs - test_size, by = 3)

for (origin in origins) {

  train <- reg_data[1:origin, ]
  test  <- reg_data[(origin + 1):(origin + test_size), ]

  model_full <- lm(
    retail_growth_pct ~ inflation_lag1 + sentiment_lag1 + retail_growth_lag1,
    data = train
  )

  # AR(1) baseline: Only lagged retail growth
  model_ar1 <- lm(
    retail_growth_pct ~ retail_growth_lag1,
    data = train
  )

  # Predict on test set
  pred_full <- predict(model_full, newdata = test)
  pred_ar1  <- predict(model_ar1,  newdata = test)

  # Compute metrics
  err_full <- test$retail_growth_pct - pred_full
  err_ar1  <- test$retail_growth_pct - pred_ar1

  rolling_results <- rbind(rolling_results, data.frame(
    origin        = origin,
    train_end     = as.character(train$date[nrow(train)]),
    # Full model metrics
    rmse_full     = sqrt(mean(err_full^2)),
    mae_full      = mean(abs(err_full)),
    r2_full       = summary(model_full)$r.squared,
    # AR(1) baseline metrics
    rmse_ar1      = sqrt(mean(err_ar1^2)),
    mae_ar1       = mean(abs(err_ar1)),
    r2_ar1        = summary(model_ar1)$r.squared,
    # Difference (positive = AR1 is better / full model is worse)
    rmse_diff     = sqrt(mean(err_full^2)) - sqrt(mean(err_ar1^2))
  ))
}


# Confidence intervals from rolling splits
ci_level <- 0.95
alpha    <- 1 - ci_level

rolling_ci <- data.frame(
  Metric = c("RMSE (Full Model)", "RMSE (AR1 Baseline)",
             "MAE (Full Model)", "MAE (AR1 Baseline)",
             "R² (Full Model)", "R² (AR1 Baseline)"),
  Mean = c(
    mean(rolling_results$rmse_full), mean(rolling_results$rmse_ar1),
    mean(rolling_results$mae_full),  mean(rolling_results$mae_ar1),
    mean(rolling_results$r2_full),   mean(rolling_results$r2_ar1)
  ),
  CI_Lower = c(
    quantile(rolling_results$rmse_full, alpha / 2),
    quantile(rolling_results$rmse_ar1,  alpha / 2),
    quantile(rolling_results$mae_full,  alpha / 2),
    quantile(rolling_results$mae_ar1,   alpha / 2),
    quantile(rolling_results$r2_full,   alpha / 2),
    quantile(rolling_results$r2_ar1,    alpha / 2)
  ),
  CI_Upper = c(
    quantile(rolling_results$rmse_full, 1 - alpha / 2),
    quantile(rolling_results$rmse_ar1,  1 - alpha / 2),
    quantile(rolling_results$mae_full,  1 - alpha / 2),
    quantile(rolling_results$mae_ar1,   1 - alpha / 2),
    quantile(rolling_results$r2_full,   1 - alpha / 2),
    quantile(rolling_results$r2_ar1,    1 - alpha / 2)
  )
) %>%
  mutate(
    Mean     = round(Mean, 4),
    CI_Lower = round(CI_Lower, 4),
    CI_Upper = round(CI_Upper, 4)
  )

kable(rolling_ci, caption = "95% Confidence Intervals from Rolling-Window CV")

Number of rolling splits: 88 





Table: 95% Confidence Intervals from Rolling-Window CV

|Metric              |   Mean| CI_Lower| CI_Upper|
|:-------------------|------:|--------:|--------:|
|RMSE (Full Model)   | 3.7767|   2.8029|   6.4934|
|RMSE (AR1 Baseline) | 3.7675|   2.8722|   6.4211|
|MAE (Full Model)    | 2.7473|   2.1357|   4.5192|
|MAE (AR1 Baseline)  | 2.7289|   2.1079|   4.4228|
|R² (Full Model)     | 0.4585|   0.3979|   0.4857|
|R² (AR1 Baseline)   | 0.4490|   0.3928|   0.4727|

In [27]:

# CHECKS IF RMSE FROM FULL MODEL IS SIGNIFICANTLY BETTER THAN AR1

paired_test_rmse <- t.test(
  rolling_results$rmse_full,
  rolling_results$rmse_ar1,
  paired = TRUE
)

paired_test_mae <- t.test(
  rolling_results$mae_full,
  rolling_results$mae_ar1,
  paired = TRUE
)

paired_summary <- data.frame(
  Metric = c("RMSE", "MAE"),
  Mean_Full = c(mean(rolling_results$rmse_full), mean(rolling_results$mae_full)),
  Mean_AR1  = c(mean(rolling_results$rmse_ar1),  mean(rolling_results$mae_ar1)),
  Mean_Diff = c(paired_test_rmse$estimate, paired_test_mae$estimate),
  t_stat    = c(paired_test_rmse$statistic, paired_test_mae$statistic),
  p_value   = c(paired_test_rmse$p.value, paired_test_mae$p.value),
  Significant = c(
    ifelse(paired_test_rmse$p.value < 0.05, "Yes", "No"),
    ifelse(paired_test_mae$p.value  < 0.05, "Yes", "No")
  )
) %>%
  mutate(
    Mean_Full = round(Mean_Full, 4),
    Mean_AR1  = round(Mean_AR1, 4),
    Mean_Diff = round(Mean_Diff, 4),
    t_stat    = round(t_stat, 4),
    p_value   = round(p_value, 4)
  )

kable(paired_summary,
      caption = "Paired t-Test: Does the Full Model Beat AR(1)?")

#takes the two RMSE columns from Method 1 and runs t.test. If p > 0.05, inflation and sentiment don't significantly beat the naive baseline.



Table: Paired t-Test: Does the Full Model Beat AR(1)?

|Metric | Mean_Full| Mean_AR1| Mean_Diff| t_stat| p_value|Significant |
|:------|---------:|--------:|---------:|------:|-------:|:-----------|
|RMSE   |    3.7767|   3.7675|    0.0091| 0.8663|  0.3887|No          |
|MAE    |    2.7473|   2.7289|    0.0183| 1.6700|  0.0985|No          |

In [26]:
library(boot)

set.seed(6230)
n_boot      <- 1000
block_size  <- 12  # 12-month blocks to preserve seasonal autocorrelation

# Full model on all data
full_model <- lm(
  retail_growth_pct ~ inflation_lag1 + sentiment_lag1 + retail_growth_lag1,
  data = reg_data
)

# Block bootstrap function
block_boot_stat <- function(data, indices_unused) {
  n <- nrow(data)
  n_blocks <- ceiling(n / block_size)

  # Sample block starting points with replacement
  block_starts <- sample(1:(n - block_size + 1), n_blocks, replace = TRUE)

  # Build bootstrap sample from consecutive blocks
  boot_indices <- unlist(lapply(block_starts, function(s) s:(s + block_size - 1)))
  boot_indices <- boot_indices[boot_indices <= n]
  boot_indices <- boot_indices[1:min(length(boot_indices), n)]

  boot_data <- data[boot_indices, ]

  model_b <- lm(
    retail_growth_pct ~ inflation_lag1 + sentiment_lag1 + retail_growth_lag1,
    data = boot_data
  )

  coefs <- coef(model_b)

  return(c(
    r_squared        = summary(model_b)$r.squared,
    intercept        = coefs["(Intercept)"],
    coef_inflation   = coefs["inflation_lag1"],
    coef_sentiment   = coefs["sentiment_lag1"],
    coef_retail_lag  = coefs["retail_growth_lag1"]
  ))
}

# Run bootstrap
boot_results <- boot(
  data      = reg_data,
  statistic = block_boot_stat,
  R         = n_boot
)

# Extract CIs
param_names <- c("R²", "Intercept", "Inflation Lag1",
                 "Sentiment Lag1", "Retail Growth Lag1")

boot_ci_table <- data.frame()
for (i in 1:5) {
  ci <- boot.ci(boot_results, index = i, type = "perc", conf = 0.95)
  boot_ci_table <- rbind(boot_ci_table, data.frame(
    Parameter = param_names[i],
    Estimate  = round(mean(boot_results$t[, i], na.rm = TRUE), 4),
    CI_Lower  = round(ci$percent[4], 4),
    CI_Upper  = round(ci$percent[5], 4),
    Spans_Zero = ifelse(
      i > 1 & ci$percent[4] <= 0 & ci$percent[5] >= 0,
      "Yes", ifelse(i == 1, "—", "No")
    )
  ))
}

kable(boot_ci_table,
      caption = "Block Bootstrap 95% CIs (1000 iterations, 12-month blocks)")

#If a coefficient CI spans zero, that variable isn't significant



Table: Block Bootstrap 95% CIs (1000 iterations, 12-month blocks)

|Parameter          | Estimate| CI_Lower| CI_Upper|Spans_Zero |
|:------------------|--------:|--------:|--------:|:----------|
|R²                 |   0.4047|   0.3036|   0.4774|—          |
|Intercept          |   0.1680|  -0.1773|   0.5840|Yes        |
|Inflation Lag1     |   0.4782|  -0.9565|   1.7244|Yes        |
|Sentiment Lag1     |   0.0802|   0.0014|   0.1520|No         |
|Retail Growth Lag1 |  -0.6370|  -0.6893|  -0.5550|No         |